# Unit 2.1: Image Filtering and Convolution

This notebook demonstrates various image filtering techniques including:
- Smoothing filters (Box, Gaussian, Median)
- Sharpening filters
- Custom convolution operations
- Padding strategies

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.ndimage import convolve
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Display Original Image

In [ ]:
# Load image
img_bgr = cv2.imread('sample.jpeg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print(f"Image Shape: {img_rgb.shape}")
print(f"Image Data Type: {img_rgb.dtype}")
print(f"Pixel Value Range: {img_gray.min()} to {img_gray.max()}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title('Original RGB Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Smoothing Filters

### 2.1 Box Filter (Moving Average)

In [ ]:
# Box Filter with different kernel sizes
kernel_sizes = [3, 5, 9]
box_filtered = []

plt.figure(figsize=(15, 4))
plt.subplot(1, 4, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

for idx, k_size in enumerate(kernel_sizes):
    # Create box kernel
    kernel = np.ones((k_size, k_size)) / (k_size ** 2)
    
    # Apply convolution
    filtered = convolve(img_gray.astype(float), kernel)
    box_filtered.append(filtered.astype(np.uint8))
    
    plt.subplot(1, 4, idx + 2)
    plt.imshow(filtered, cmap='gray')
    plt.title(f'Box Filter {k_size}x{k_size}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print("Box Filter kernels created for sizes: 3x3, 5x5, 9x9")

### 2.2 Gaussian Filter

In [ ]:
# Gaussian Filter with different sigma values
sigmas = [0.5, 1.0, 2.0]
gaussian_filtered = []

plt.figure(figsize=(15, 4))
plt.subplot(1, 4, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

for idx, sigma in enumerate(sigmas):
    # Apply Gaussian blur using OpenCV
    filtered = cv2.GaussianBlur(img_gray, (5, 5), sigma)
    gaussian_filtered.append(filtered)
    
    plt.subplot(1, 4, idx + 2)
    plt.imshow(filtered, cmap='gray')
    plt.title(f'Gaussian σ={sigma}')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Show Gaussian kernel
def gaussian_kernel(size, sigma):
    kernel = np.fromfunction(lambda x, y: (1/(2*np.pi*sigma**2)) * np.exp(-((x-size//2)**2 + (y-size//2)**2)/(2*sigma**2)), 
                           (size, size))
    return kernel / kernel.sum()

g_kernel = gaussian_kernel(5, 1.0)
print("\nGaussian Kernel (5x5, σ=1.0):")
print(g_kernel)

### 2.3 Median Filter

In [ ]:
# Add salt and pepper noise first
def add_salt_pepper_noise(image, salt_prob=0.02, pepper_prob=0.02):
    noisy = image.copy().astype(float)
    
    # Add salt (white noise)
    salt_mask = np.random.rand(*image.shape) < salt_prob
    noisy[salt_mask] = 255
    
    # Add pepper (black noise)
    pepper_mask = np.random.rand(*image.shape) < pepper_prob
    noisy[pepper_mask] = 0
    
    return noisy.astype(np.uint8)

# Create noisy image
noisy_img = add_salt_pepper_noise(img_gray)

# Apply median filter with different kernel sizes
kernel_sizes = [3, 5, 7]
median_filtered = []

plt.figure(figsize=(15, 4))
plt.subplot(1, 4, 1)
plt.imshow(noisy_img, cmap='gray')
plt.title('Noisy Image')
plt.axis('off')

for idx, k_size in enumerate(kernel_sizes):
    filtered = cv2.medianBlur(noisy_img, k_size)
    median_filtered.append(filtered)
    
    plt.subplot(1, 4, idx + 2)
    plt.imshow(filtered, cmap='gray')
    plt.title(f'Median {k_size}x{k_size}')
    plt.axis('off')

plt.tight_layout()
plt.show()

## 3. Sharpening Filters

In [ ]:
# 3.1 Laplacian Sharpening
laplacian_kernel = np.array([[0, 1, 0],
                              [1, -4, 1],
                              [0, 1, 0]], dtype=np.float32)

laplacian = cv2.filter2D(img_gray.astype(np.float32), -1, laplacian_kernel)
laplacian_normalized = np.clip(laplacian, 0, 255).astype(np.uint8)

# 3.2 Unsharp Mask (Sharpening by subtraction)
gaussian_blur = cv2.GaussianBlur(img_gray, (5, 5), 1.0)
unsharp_mask = cv2.subtract(img_gray.astype(np.float32), gaussian_blur.astype(np.float32))
sharpened = np.clip(img_gray.astype(np.float32) + unsharp_mask * 1.5, 0, 255).astype(np.uint8)

# Display comparison
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(laplacian_normalized, cmap='gray')
plt.title('Laplacian Sharpening')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(sharpened, cmap='gray')
plt.title('Unsharp Mask (α=1.5)')
plt.axis('off')

plt.tight_layout()
plt.show()

print("Sharpening kernels applied successfully")

## 4. Custom Convolution Operation

In [ ]:
def manual_convolution(image, kernel, padding='zero'):
    """
    Manual 2D convolution implementation
    
    Args:
        image: Input image (2D numpy array)
        kernel: Convolution kernel (2D numpy array)
        padding: 'zero', 'reflect', 'replicate'
    
    Returns:
        Convolved image
    """
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape
    
    # Pad image
    pad_h, pad_w = k_h // 2, k_w // 2
    
    if padding == 'zero':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    elif padding == 'reflect':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    else:  # replicate
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    
    # Initialize output
    output = np.zeros_like(image, dtype=np.float32)
    
    # Perform convolution
    for i in range(img_h):
        for j in range(img_w):
            region = padded[i:i+k_h, j:j+k_w]
            output[i, j] = np.sum(region * kernel)
    
    return output

# Test with different kernels
edge_kernel = np.array([[-1, -1, -1],
                         [-1,  8, -1],
                         [-1, -1, -1]], dtype=np.float32)

# Apply manual convolution
small_img = cv2.resize(img_gray, (150, 150))  # Use smaller image for faster computation
custom_conv = manual_convolution(small_img.astype(np.float32), edge_kernel, padding='zero')
custom_conv_normalized = np.clip(custom_conv, 0, 255).astype(np.uint8)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(small_img, cmap='gray')
plt.title('Original (Resized)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(edge_kernel, cmap='hot')
plt.title('Edge Detection Kernel')
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(custom_conv_normalized, cmap='gray')
plt.title('Manual Convolution Result')
plt.axis('off')

plt.tight_layout()
plt.show()

print("Manual convolution implementation completed")

## 5. Padding Strategies Comparison

In [ ]:
# Create a small test image to visualize padding
test_img = np.array([[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 9]], dtype=np.float32)

# Apply different padding strategies
zero_pad = np.pad(test_img, 1, mode='constant', constant_values=0)
reflect_pad = np.pad(test_img, 1, mode='reflect')
replicate_pad = np.pad(test_img, 1, mode='edge')
circular_pad = np.pad(test_img, 1, mode='wrap')

plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(zero_pad, cmap='tab10')
plt.title('Zero Padding')
for i in range(zero_pad.shape[0]):
    for j in range(zero_pad.shape[1]):
        plt.text(j, i, f'{int(zero_pad[i,j])}', ha='center', va='center', color='white')

plt.subplot(1, 4, 2)
plt.imshow(reflect_pad, cmap='tab10')
plt.title('Reflection Padding')
for i in range(reflect_pad.shape[0]):
    for j in range(reflect_pad.shape[1]):
        plt.text(j, i, f'{int(reflect_pad[i,j])}', ha='center', va='center', color='white')

plt.subplot(1, 4, 3)
plt.imshow(replicate_pad, cmap='tab10')
plt.title('Replication Padding')
for i in range(replicate_pad.shape[0]):
    for j in range(replicate_pad.shape[1]):
        plt.text(j, i, f'{int(replicate_pad[i,j])}', ha='center', va='center', color='white')

plt.subplot(1, 4, 4)
plt.imshow(circular_pad, cmap='tab10')
plt.title('Circular Padding')
for i in range(circular_pad.shape[0]):
    for j in range(circular_pad.shape[1]):
        plt.text(j, i, f'{int(circular_pad[i,j])}', ha='center', va='center', color='white')

plt.tight_layout()
plt.show()

print("All padding strategies demonstrated")

## 6. Filter Performance Metrics

In [ ]:
# Calculate image statistics for filtered images
from sklearn.metrics import mean_squared_error

print("Image Statistics After Filtering\n")
print("Filter Type\t\tMean(Intensity)\tStd Dev\tMin Value\tMax Value")
print("-" * 80)

print(f"Original\t\t{img_gray.mean():.2f}\t\t{img_gray.std():.2f}\t{img_gray.min()}\t\t{img_gray.max()}")
print(f"Box Filter (5x5)\t{box_filtered[1].mean():.2f}\t\t{box_filtered[1].std():.2f}\t{box_filtered[1].min()}\t\t{box_filtered[1].max()}")
print(f"Gaussian (σ=1.0)\t{gaussian_filtered[1].mean():.2f}\t\t{gaussian_filtered[1].std():.2f}\t{gaussian_filtered[1].min()}\t\t{gaussian_filtered[1].max()}")
print(f"Median (5x5)\t\t{median_filtered[1].mean():.2f}\t\t{median_filtered[1].std():.2f}\t{median_filtered[1].min()}\t\t{median_filtered[1].max()}")
print(f"Sharpened\t\t{sharpened.mean():.2f}\t\t{sharpened.std():.2f}\t{sharpened.min()}\t\t{sharpened.max()}")

## Summary

### Key Takeaways:
1. **Box Filters**: Simple but can cause blurring
2. **Gaussian Filters**: Better edge preservation than box filters
3. **Median Filters**: Excellent for salt & pepper noise removal
4. **Sharpening Filters**: Enhance edges and details
5. **Padding Strategies**: Important for edge pixel handling
6. **Convolution**: Foundation of all filtering operations

### Next Steps:
- Study edge detection methods (Sobel, Canny, etc.)
- Explore frequency domain filtering
- Implement custom filters for specific applications